In [ ]:

import scanpy as sc
import os

file_id = "adataFinalPlusOneAftereSCVIRecluster_clean.h5ad"
UPLOAD_DIR = "../../../persistent01"

file_path = os.path.join(UPLOAD_DIR, file_id)

adata = sc.read_h5ad(file_path)

adata


In [ ]:

uns_key = "leiden_1_0_test1"

rgg = adata.uns["rank_genes_groups"]

groups = rgg["names"].dtype.names

top_genes = set()

for g in groups:
  top_genes.update(rgg["names"][g][:4])

top_genes


In [ ]:

dp = sc.pl.DotPlot(adata, groupby=uns_key, var_names=sorted(top_genes))

mean_expr_df = dp.dot_color_df # mean expression
frac_expr_df = dp.dot_size_df # fraction expression

# print(mean_expr_df)
# print(frac_expr_df)

# adata.file.close()

combined_df = mean_expr_df.stack().to_frame("mean_expr").join(
  frac_expr_df.stack().to_frame("frac_expr")
).reset_index().rename(columns={"level_0": "group", "level_1": "gene"})

combined_df



In [ ]:

sorted_df = combined_df.sort_values(["mean_expr", "gene"], ascending=False)

sorted_df

In [ ]:

# filtered_df = sorted_df.groupby("gene").head(1)
filtered_df = sorted_df.drop_duplicates("gene", keep="first")

filtered_df


In [ ]:

genes = filtered_df["gene"].head(10)

genes


In [ ]:

filtered_df = filtered_df.drop_duplicates("group", keep="first")

filtered_df



In [ ]:

sorted_df = combined_df.sort_values(["mean_expr", "gene"], ascending=False)

filtered_df = sorted_df.groupby("gene").head(1)
filtered_df = filtered_df.drop_duplicates("group", keep="first")

# filtered_df

genes = filtered_df["gene"].head(10)

genes

# groups = filtered_df["group"]


In [ ]:

groups = filtered_df["group"].head(4)

groups

In [ ]:

results_df = sorted_df[sorted_df["gene"].isin(genes) & sorted_df["group"].isin(groups) ]

results_df

# results_df.sort_values

# results_df["gene"].drop_duplicates()


In [ ]:

results_df.to_dict(orient="records")

In [ ]:

# sc.pl.rank_genes_groups_dotplot(adata, groupby=uns_key, n_genes=20)
sc.pl.rank_genes_groups_dotplot(adata, groupby=uns_key, var_names=sorted(top_genes))